In [1]:

from datasets import Dataset
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
from datasets import Dataset
import evaluate
import torch
import numpy as np
from transformers import DataCollatorForSeq2Seq
from transformers import Trainer, TrainingArguments, BitsAndBytesConfig
from transformers import StoppingCriteria, StoppingCriteriaList
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
import argparse
import os
import re

In [ ]:

from constants import COLLECTIVE_TXT, RESPONSE_TXT, PROPERTY_TXT, CONCLUSION_TXT, MAX_LENGTH
from load_dataset import load_asohmo
from utils import print_trainable_parameters

In [2]:
print(torch.cuda.is_available())
print(torch.version.cuda)

False
12.4


In [3]:
# Define the arguments
device = torch.device("cuda")
parser = argparse.ArgumentParser(description="Train models and generate counter-narratives against hate speech")
parser.add_argument("--generation_strategy", type=str, choices=["zeroshot", "fewshot", "finetuned", "finetuning"])
parser.add_argument("--use_extra_info", type=str, choices=["collective", "premises", "all", ""], default="")
parser.add_argument("--cn_strategy", type=str, default="", choices=["a", "b", "c", ""])
parser.add_argument("--model_name", type=str, default="google/flan-t5-base")
parser.add_argument("--quantized", type=bool, default=False)
parser.add_argument("--load_from_hub", type=bool, default=False)
parser.add_argument("--fewshot_examples", type=int, default=5)
parser.add_argument("--short", type=bool, default=False)

# Simulate the CLI args in Jupyter
test_args = [
    "--generation_strategy", "finetuning",
    "--use_extra_info", "all",
    "--cn_strategy", "a",
    "--model_name", "google/flan-t5-base",
    "--quantized", "False",
    "--load_from_hub", "False",
    "--fewshot_examples", "5",
    "--short", "False"
]
args = parser.parse_args(test_args)


In [4]:
model_name = args.model_name
language = "spanish"
dataset = ""
finetuning = args.generation_strategy == "finetuning"
is_causallm = "flan-t5" not in model_name
model_without_user_interface = "tiiuae/falcon" in model_name
extra_info = args.use_extra_info if args.use_extra_info != "" else "no-info"
cn_strategy = args.cn_strategy if args.cn_strategy != "" else "no-strategy"
repository_id = f"{model_name.split('/')[1]}_{language}_{extra_info}_{cn_strategy}"
SHORT = args.short


In [5]:
def add_arg_info(extra_info_sample, tweet):
    if "collective" in extra_info_sample and extra_info_sample["collective"].strip() != "":
        collective = extra_info_sample["collective"]
        tweet += f" | {COLLECTIVE_TXT}{collective}"
    if "property" in extra_info_sample and extra_info_sample["property"].strip() != "":
        propert = extra_info_sample["property"]
        tweet += f" | {PROPERTY_TXT}{propert}"
    if "justification" in extra_info_sample and extra_info_sample["justification"].strip() != "":
        justification = extra_info_sample["justification"]
        tweet += f" | {CONCLUSION_TXT}{justification}"
    if "conclusion" in extra_info_sample and extra_info_sample["conclusion"].strip() != "":
        conclusion = extra_info_sample["conclusion"]
        tweet += f" | {CONCLUSION_TXT}{conclusion}"

In [20]:
def generate_prompt(text, extra_info, fewshot_examples, short=False):
    initial_prompt = "Sos un operario de una ONG experto en generación de contra-narrativas y contra-discurso contra el discurso de odio. Solo hablas Español y eres incapáz de generar texto en otro idioma. Tu tarea es generar una respuesta en Español a un tweet de odio."
    argumentative_prompt = " También se te proporcionará información argumentativa sobre el tweet separada por '|'. Utiliza esta información para elaborar una respuesta. No incluyas esta información en la respuesta."
    final_prompt = " Responde el tweet directamente sin agregar ninguna otra información que no sea la respuesta. El tweet con el mensaje de odio es el siguiente:\n\n"
    has_arg_info = "collective" in extra_info or "property" in extra_info or "justification" in extra_info or "conclusion" in extra_info
    if model_without_user_interface or not is_causallm:
        if short:
                prompt = "Escribe una respuesta en contra del odio para el siguiente tweet de odio: "
        else:
            prompt = initial_prompt
            if has_arg_info:
                prompt += argumentative_prompt
            prompt += final_prompt
        if fewshot_examples:
            #TODO: Move this to a separate function because its repeated in the other case
            counter = 0
            for example in fewshot_examples:
                if counter == args.fewshot_examples:
                    break
                else:
                    counter += 1
                tweet = example["hateSpeech"]
                cn = example["counterSpeech"]
                extra_info_sample = example["extra_info"]
                tweet = add_arg_info(extra_info_sample, tweet)
                prompt += f"'{tweet}'{RESPONSE_TXT}'{cn}'\n"
        text = add_arg_info(extra_info, text)
        prompt += f"'{text}'{RESPONSE_TXT} "
    else:
        prompt = [{"role": "system", "content": initial_prompt}]
        if fewshot_examples:
            counter = 0
            for example in fewshot_examples:
                if counter == args.fewshot_examples:
                    break
                else:
                    counter += 1
                tweet = example["hateSpeech"]
                cn = example["counterSpeech"]
                extra_info_sample = example["extra_info"]
                tweet = add_arg_info(extra_info_sample, tweet)
                prompt.append({"role": "user", "content": tweet})
                prompt.append({"role": "assistant", "content": cn})
        text = add_arg_info(extra_info, text)
        prompt.append({"role": "user", "content": text})
    return prompt

In [23]:
# Load dataset
print("loading asohmo...")
exxtra_info = args.use_extra_info
if args.cn_strategy == "a":
    if args.use_extra_info == "premises":
        exxtra_info = "cn_a"
    else:
        exxtra_info = "cn_a_no_info"
elif args.cn_strategy == "b":
    if args.use_extra_info == "collective":
        exxtra_info = "cn_b"
    else:
        exxtra_info = "cn_b_no_info"
elif args.cn_strategy == "c":
    if args.use_extra_info == "premises":
        exxtra_info = "cn_c"
    else:
        exxtra_info = "cn_c_no_info"
datasetss = load_asohmo(use_extra_info=exxtra_info, finetuning)

loading asohmo
12 - 33 - 7
Counter narratives without the required type of counter-narrative: 4
Non arg examples discarted for not having CN: 52
27 - 101 - 12


In [24]:
test_dataset = datasetss[0]
test_data = Dataset.from_pandas(pd.DataFrame(test_dataset))

In [25]:
train_dataset = datasetss[1]
val_dataset = datasetss[2]
train_data = Dataset.from_pandas(pd.DataFrame(train_dataset + val_dataset))

In [26]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
if "Mistral" in model_name or "Mixtral" in model_name or "Qwen" in model_name:
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        padding_side="left",
        add_eos_token=True,
        add_bos_token=True,
)
if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token

if args.generation_strategy == "finetuned":
    # if args.cn_strategy != "":
    finetuned_name = f"{args.model_name.split('/')[-1]}_{language}_{extra_info}_{cn_strategy}"
    if args.load_from_hub:
        model_name = f"CounterNarratives/{finetuned_name}"
    else:
        model_name = f"pretrained_models/{finetuned_name}"
    print("LOADING MODEL: ", model_name)
    # else:
    #     model_name = f"pretrained_models/{dataset}_{args.model_name.replace('/', '-')}_multi_{args.use_extra_info}_2e-05_8Epochs"


In [27]:
if args.quantized:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    if is_causallm:
        model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, resume_download=True)
    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name, quantization_config=bnb_config)
    if args.generation_strategy == "finetuning":
        model.gradient_checkpointing_enable()
        model = prepare_model_for_kbit_training(model)
        config = LoraConfig(
            r=32,
            lora_alpha=64,
            target_modules=[
                "q_proj",
                "k_proj",
                "v_proj",
                "o_proj",
                "gate_proj",
                "up_proj",
                "down_proj",
                "lm_head",
            ],
            bias="none",
            lora_dropout=0.05,  # Conventional
            task_type=TaskType.CAUSAL_LM,
        )
        model = get_peft_model(model, config)
        print_trainable_parameters(model)
else:
    if is_causallm:
        model = AutoModelForCausalLM.from_pretrained(model_name)
    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        model.resize_token_embeddings(len(tokenizer))
    model.to(device)

AttributeError: 'frozenset' object has no attribute 'discard'

In [34]:
print("model", model)

NameError: name 'model' is not defined

In [28]:
datasett = test_dataset
if finetuning:
    datasett += train_dataset
    datasett += val_dataset

In [29]:
def preprocess(sample, padding="max_length", is_testing = False, fewshot_examples = None):
    inputs = generate_prompt(sample["hateSpeech"], sample["language"], sample["extra_info"], fewshot_examples, short=SHORT)
    # ES NECESARIO ESTO?
    if finetuning:
        if is_causallm:
            if model_without_user_interface:
                model_inputs = tokenizer(inputs + sample["counterSpeech"], padding=padding, max_length=MAX_LENGTH, truncation=True)
            else:
                inputs.append({"role": "assistant", "content": sample["counterSpeech"]})
                input_with_chat_template = tokenizer.apply_chat_template(inputs, return_tensors="pt", tokenize=False, add_generation_prompt=False)
                model_inputs = tokenizer(input_with_chat_template, padding=padding, max_length=MAX_LENGTH, truncation=True)
            model_inputs["labels"] = model_inputs["input_ids"].copy()

        else:
            model_inputs = tokenizer(inputs, padding=padding, max_length=MAX_LENGTH, truncation=True)
            labels = tokenizer(sample["counterSpeech"], padding=padding, max_length=MAX_LENGTH, truncation=True)
            if padding == "max_length":
                labels["input_ids"] = [
                    (l if l != tokenizer.pad_token_id else -100) for l in labels["input_ids"]
                ]
            model_inputs["labels"] = labels["input_ids"]
    else:
        if model_without_user_interface or not is_causallm:
            model_inputs = tokenizer(inputs, padding=padding, max_length=MAX_LENGTH, truncation=True, return_tensors="pt")
        else:
            input_with_chat_template = tokenizer.apply_chat_template(inputs, return_tensors="pt", add_generation_prompt=True, tokenize = False)
            model_inputs = tokenizer(input_with_chat_template, padding=padding, max_length=MAX_LENGTH, truncation=True, return_tensors="pt")

    if is_testing:
        test_inputs = {"example": model_inputs, "counterSpeech": sample["counterSpeech"], "number": sample["extra_info"]["number"], "tweet": sample["hateSpeech"]}
        return test_inputs
    return model_inputs

In [30]:
class StoppingCriteriaSub(StoppingCriteria):
    def __init__(self, stops = [], encounters=1):
        super().__init__()
        self.stops = [stop.to("cuda") for stop in stops]

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor):
        last_token = input_ids[0][-1]
        for stop in self.stops:
            if tokenizer.decode(stop) == tokenizer.decode(last_token):
                return True
        return False


stop_words = [".", "]", "']", "']\n", "\n", "]\n", "\n\n", "']\n\n", "</s>"]
stop_words_ids = [tokenizer(stop_word, return_tensors='pt', add_special_tokens=False)['input_ids'].squeeze() for stop_word in stop_words]
stopping_criteria = StoppingCriteriaList([StoppingCriteriaSub(stops=stop_words_ids)])


RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

In [31]:
def evaluate_generation(testing_datasets, top_sampling=False, beam_search=True, temperature=False, is_causallm = True):

    # sbert_avg = 0.0

    # sbert = SentenceTransformer('all-MiniLM-L6-v2')

    generation_strategy = args.generation_strategy + "-" + str(args.fewshot_examples) if args.generation_strategy == "fewshot" else args.generation_strategy
    foldername = f"{dataset}_{args.model_name}_{language}_{generation_strategy}_{args.use_extra_info}_{args.cn_strategy}_{top_sampling}_{beam_search}_{temperature}".replace("/", "-")
    if not os.path.exists(f"test_results/{foldername}"):
        os.makedirs(f"test_results/{foldername}")
    for example in testing_datasets:
        inputt = example["example"]
        tweet = example["tweet"]
        number = example["number"]
        w = open(f"test_results/{foldername}/hate_tweet_{language}_{number}", 'w')
        inputt.to(device)
        if beam_search:
            result = model.generate(**inputt, do_sample=True, max_new_tokens=280, num_beams=4, no_repeat_ngram_size=4, num_return_sequences=1, eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id)
        elif top_sampling:
            result = model.generate(inputs=inputt, do_sample=True, max_new_tokens=280, top_k=0, top_p=0.92, no_repeat_ngram_size=2, num_return_sequences=1, stopping_criteria=stopping_criteria, eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id)
        elif temperature:
            result = model.generate(inputs=inputt, do_sample=True, max_new_tokens=280, temperature=0.7, no_repeat_ngram_size=2, num_return_sequences=1, stopping_criteria=stopping_criteria, eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id)
        else:
            result = model.generate(inputs=inputt, max_new_tokens=280, no_repeat_ngram_size=2, num_return_sequences=1, stopping_criteria=stopping_criteria, eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id)
        if is_causallm:
            preds = str(tokenizer.decode(result[0][len(inputt["input_ids"][0]):]))
        else:
            preds = str(tokenizer.decode(result[0]))
            preds = re.sub(r'<extra_id_\d*>', '', preds)
        response = preds.replace("\n", "").replace("\t", "").split("[/INST]")[-1].replace("<s>", "").replace("</s>", "").replace("<pad>", "").replace("<|eot_id|>", "").replace("<|start_header_id|>", "").replace("<|im_end|>", "")
        print("----------------------------------tweet-----------------------------")
        print(tweet)
        print("----------------------------------preds-----------------------------")
        print(response)
        print("\n")
        # for labels in example[0]["counterSpeech"]:

            # cosine_scores_preds = sbert.encode([preds], convert_to_tensor=True)
            # cosine_scores_labels = sbert.encode([labels], convert_to_tensor=True)

            # sbert_score = util.cos_sim(cosine_scores_preds, cosine_scores_labels)

            # sbert_avg += sbert_score[0][0].item()
        w.write(tweet.replace("\n", "").replace("\t", ""))
        w.write('\n')
        w.write(response)
        w.write("\n")
        w.close()

In [32]:
train_data = train_data.map(preprocess)

Map:   0%|          | 0/113 [00:00<?, ? examples/s]

In [33]:
if finetuning:

    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        # Replace -100 in the labels as we can't decode them.
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        # decoded_inputs = tokenizer.batch_decode(inputs, skip_special_tokens=True)

        # Some simple post-processing
        # decoded_preds, decoded_labels, decoded_inputs = postprocess_text(decoded_preds, decoded_labels, decoded_inputs)
        metric = evaluate.load("bertscore")
        # Using rouge score
        result = metric.compute(predictions=decoded_preds, references=decoded_labels)


        # result = {k: round(v * 100, 4) for k, v in result.items()}
        prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
        result["gen_len"] = np.mean(prediction_lens)
        result["prediction"] = decoded_preds
        result["labels"] = decoded_labels
        # result["inputs"] = decoded_inputs
        return result

    # we want to ignore tokenizer pad token in the loss
    label_pad_token_id = -100
    # Data collator
    data_collator = DataCollatorForSeq2Seq(
        tokenizer,
        model=model,
        label_pad_token_id=label_pad_token_id,
    )

    optimizer = "adamw_bnb_8bit" if args.quantized else "adamw_torch"
    # Define training args
    training_args = TrainingArguments(
        output_dir=repository_id,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        # predict_with_generate=True,
        # generation_max_length=200,
        # generation_num_beams=4,
        # TODO: turn this on and check if it works
        fp16=False, # Overflows with fp16
        learning_rate=2e-04,
        num_train_epochs=8,
        lr_scheduler_type="cosine",
        # include_inputs_for_metrics=True,
        # logging & evaluation strategies
        # logging_dir=f"{repository_id}/logs",
        # logging_strategy="steps",
        # logging_steps=5,
        optim=optimizer,
        # evaluation_strategy="epoch",
        # save_strategy="epoch",
        # save_total_limit=10,
        # load_best_model_at_end=True,
        # metric_for_best_model="overall_f1",
        # push to hub parameters
        # report_to="tensorboard",
        # push_to_hub=True,
        # hub_strategy="every_save",
        # hub_model_id=repository_id,
    )

    # Create Trainer instance
    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=data_collator,
        train_dataset=train_data,
        # eval_dataset=val_data,
        # compute_metrics=compute_metrics,
    )

    model.config.use_cache = False  # silence the warnings. Please re-enable for inference!
    trainer.train()

    if not os.path.exists("pretrained_models"):
        os.makedirs("pretrained_models")
    model.save_pretrained("pretrained_models/" + repository_id)

    model.push_to_hub("CounterNarratives/" + repository_id)

else:
    fewshot_examples = train_data if args.generation_strategy == "fewshot" else None
    preprocessed_dataset = []
    for example in test_data:
        preprocessed_dataset.append(preprocess(example, is_testing = True, fewshot_examples = fewshot_examples))

    print("generating")
    # evaluate_generation(preprocessed_dataset)
    # evaluate_generation(preprocessed_dataset, top_sampling=True)
    # evaluate_generation(preprocessed_dataset, temperature=True)
    evaluate_generation(preprocessed_dataset, beam_search=True, is_causallm=is_causallm)

NameError: name 'model' is not defined